# Text Summarization using TF-IDF
## Manchester City Article - Extractive Summarization

**Objective:** Generate an extractive summary of the Manchester City Club World Cup article using TF-IDF scoring methodology.

**Method:** 
- Extract sentences based on their TF-IDF importance scores
- Use English stopwords for preprocessing
- Calculate threshold as the average of all sentence scores
- Select sentences with score >= threshold

**Output:** A concise summary preserving the original sentence order

## Step 1: Import Required Libraries

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("✓ All libraries imported successfully")

## Step 2: Define the Document (Manchester City Article)

In [ ]:
sentence = """
Manchester City makes history by winning Club World Cup

Manchester City capped off its incredible year with yet another trophy, dismantling Fluminense 4-0 to win the Club World Cup on Friday.

Having already won the Premier League, Champions League, FA Cup and Super Cup, Pep Guardiola's side now boasts five trophies this calendar year, becoming the first English club to ever hold all those titles simultaneously.

The final piece of the jigsaw came on a highly charged night in Saudi Arabia as Manchester City outclassed its Brazilian opponents.

"We've shown over the past 12 months we are the best team in the world. Our results prove that and the consistency we have managed has been amazing," club captain Kyle Walker said after the game, per Sky Sports.

"To win these five trophies – for me, the five biggest prizes available to us – is incredible. I am so proud to have been a part of this and I can honestly say it's an honour to play alongside these players. I couldn't ask for better teammates."

It took just 40 seconds for Manchester City to take the lead.

Brazilian left-back Marcelo miscued a pass in the opening exchanges which let Nathan Aké free to shoot from distance. The defender's effort cannoned back off the post but forward Julián Álvarez was in the right place to turn the rebound into the net with his chest.

City continued to look dangerous and doubled its lead before the break after Phil Foden's attempted cross was deflected into his own net by Fluminense defender Nino.

Foden then got on the scoresheet himself in the 72nd minute after a prodding home from close range.

The rout was completed in the 88th minute when Álvarez capped off a brilliant performance with a clinical finish into the far corner.

City's defence was largely untested for during the game, underlining the team's dominance during this unforgettable year.

"As a manager it is what I am most proud of; that we are always there. No matter how much we win, no matter what trophies we lift, we are there again to fight for the next one," City boss Guardiola said after the match, according to Sky Sports.

"To win the Treble was truly special, but to win two more trophies and now hold these five major titles shows the unique mentality of this team, of the Club and its fans.

"It is something no other English team has ever achieved, and we will always remember this incredible time we spent together."

The game ended in some unsavoury scenes as a scuffle broke out between players on the pitch after the final whistle, but the game will be remembered as yet another successful night for City.

The champion heads back to England where it faces a tough title defence in the Premier League.

It currently sits fourth in the table and will face Everton in its next fixture on Wednesday.
"""

print("✓ Document loaded")
print(f"Total characters: {len(sentence)}")

## Step 3: Preprocess - Sentence Tokenization

In [ ]:
print("="*80)
print("STEP 1: SENTENCE TOKENIZATION")
print("="*80)

sent_token = sent_tokenize(sentence)
print(f"\nTotal sentences extracted: {len(sent_token)}\n")

for idx, sent in enumerate(sent_token, 1):
    print(f"{idx:2d}. {sent[:75]}...")

## Step 4: TF-IDF Feature Extraction

In [ ]:
print("\n" + "="*80)
print("STEP 2: TF-IDF VECTORIZATION (English Stopwords)")
print("="*80)

vectorizer = TfidfVectorizer(stop_words='english')
features = vectorizer.fit_transform(sent_token)

print(f"\nFeature Matrix Shape: {features.shape}")
print(f"  - Number of sentences: {features.shape[0]}")
print(f"  - Number of unique words (features): {features.shape[1]}")

feature_names = vectorizer.get_feature_names_out()
print(f"\nTop 20 Important Words in Vocabulary:")

# Get top words by average TF-IDF
avg_tfidf = features.mean(axis=0).A1
top_indices = avg_tfidf.argsort()[-20:][::-1]
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. {feature_names[idx]:15s} - {avg_tfidf[idx]:.4f}")

## Step 5: Calculate Sentence Scores

In [ ]:
print("\n" + "="*80)
print("STEP 3: CALCULATE AVERAGE TF-IDF SCORE FOR EACH SENTENCE")
print("="*80)

sent_scores = []
print("\nSentence Scores:")
print("-" * 80)

for idx, tfidf_vector in enumerate(features, 1):
    sent_score = tfidf_vector.sum()
    sent_length = len(tfidf_vector.data)
    avg_score = sent_score / sent_length if sent_length > 0 else 0
    sent_scores.append(avg_score)
    print(f"Sentence {idx:2d}: {avg_score:.4f}  (sum={sent_score:.4f}, words={sent_length})")

# Calculate threshold
threshold = sum(sent_scores) / len(sent_scores)
print("\n" + "-"*80)
print(f"THRESHOLD (Average Score): {threshold:.4f}")
print("-"*80)

## Step 6: Visualize Sentence Scores

In [ ]:
print("\nGenerating visualization...\n")

plt.figure(figsize=(15, 6))
colors = ['#2ecc71' if score >= threshold else '#e74c3c' for score in sent_scores]
bars = plt.bar(range(1, len(sent_scores) + 1), sent_scores, color=colors, edgecolor='black', linewidth=1.2)

# Add threshold line
plt.axhline(y=threshold, color='#3498db', linestyle='--', linewidth=2.5, label=f'Threshold: {threshold:.4f}')

plt.xlabel("Sentence Number", fontsize=12, fontweight='bold')
plt.ylabel("Average TF-IDF Score", fontsize=12, fontweight='bold')
plt.title("Average TF-IDF Score per Sentence\nManchester City Article", fontsize=14, fontweight='bold')
plt.xticks(range(1, len(sent_scores) + 1), fontsize=10)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.legend(fontsize=11, loc='upper right')

# Add color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='Selected (≥ threshold)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Not selected (< threshold)')
]
plt.legend(handles=legend_elements + [plt.Line2D([0], [0], color='#3498db', linewidth=2.5, linestyle='--')], 
           labels=['Selected (≥ threshold)', 'Not selected (< threshold)', f'Threshold: {threshold:.4f}'],
           fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Step 7: Generate Extractive Summary

In [ ]:
print("\n" + "="*80)
print("STEP 4: GENERATE EXTRACTIVE SUMMARY")
print("="*80)

final_summ = ""
summary_indices = []

print(f"\n**Sentences Selected for Summary (Score >= {threshold:.4f}):**\n")
print("-" * 80)

for idx, score in enumerate(sent_scores, 1):
    if score >= threshold:
        summary_indices.append(idx)
        final_summ = final_summ + " " + sent_token[idx-1]
        print(f"✓ [{idx:2d}] {sent_token[idx-1][:65]}...")
    else:
        print(f"✗ [{idx:2d}] {sent_token[idx-1][:65]}...")

print("\n" + "-"*80)

## Step 8: Display Final Summary & Statistics

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY - MANCHESTER CITY ARTICLE")
print("="*80)

total_original_words = sum(len(s.split()) for s in sent_token)
total_summary_words = len(final_summ.split())
compression_ratio = (total_summary_words / total_original_words * 100)

print(f"\n📊 STATISTICS:")
print(f"  • Original text: {total_original_words} words")
print(f"  • Summary: {total_summary_words} words")
print(f"  • Compression ratio: {compression_ratio:.1f}%")
print(f"  • Sentences selected: {len(summary_indices)}/{len(sent_token)}")
print(f"  • Sentence reduction: {((1 - len(summary_indices)/len(sent_token))*100):.1f}%")

print("\n" + "-"*80)
print("\n📝 SUMMARY TEXT:\n")
print(final_summ.strip())
print("\n" + "="*80)

## Step 9: Detailed Analysis Table

In [ ]:
print("\n📋 DETAILED SENTENCE ANALYSIS:\n")

summary_df = pd.DataFrame({
    'Sent #': range(1, len(sent_token) + 1),
    'Score': [f'{s:.4f}' for s in sent_scores],
    'Selected': ['✓' if i+1 in summary_indices else '✗' for i in range(len(sent_token))],
    'Status': ['INCLUDED' if i+1 in summary_indices else 'EXCLUDED' for i in range(len(sent_token))],
    'Text Preview': [s[:55] + '...' if len(s) > 55 else s for s in sent_token]
})

print(summary_df.to_string(index=False))
print("\n" + "="*80)

## Summary Algorithm Explanation

### How This Works:

1. **Tokenization**: Text split into individual sentences
2. **TF-IDF Calculation**:
   - TF (Term Frequency): How often a word appears in each sentence
   - IDF (Inverse Document Frequency): How unique/important a word is across all sentences
   - TF-IDF = TF × IDF (combines both measures)

3. **Sentence Scoring**: Average TF-IDF score of all words in each sentence
4. **Threshold**: Average of all sentence scores
5. **Selection**: Sentences with score ≥ threshold are selected
6. **Summary**: Selected sentences arranged in original order

### Key Insights:

- ✓ **Green bars** = Sentences important enough to include
- ✗ **Red bars** = Sentences with less important information
- **Blue dashed line** = Decision boundary (threshold)

### Advantages of This Method:

- ✓ Fast and efficient
- ✓ No training data required
- ✓ Preserves original sentence order
- ✓ Works well for news articles
- ✗ Cannot paraphrase or compress individual sentences
- ✗ 
 only (selects existing sentences, doesn't create new ones)